In [ ]:
import pandas as pd

pd.set_option('display.max_rows', 500)

from config import DATA_VERSION_IDENTIFIER, SAVE_FILES, SELECTED_VARIABLES, MUTATION_LABELS_PATH_BM
from core.dummy import save_with_dummy

In [ ]:
MUTATION_INPUT_FILEPATH = "~/mounts/research/husdatalake/disease/scripts/AML_mutation_prediction/clinical_input_data/KI_mutations.csv"
BM_VARIABLE_TABLE_PATH = MUTATION_LABELS_PATH_BM

### **Mutation labels**

In [ ]:
df = pd.read_csv(MUTATION_INPUT_FILEPATH)

print(f"Number of available samples: {len(df)}")

# get existing mutation variables in Sweden-KI dataset
existing_variables = [variable for variable in SELECTED_VARIABLES if variable in df.columns]

mutation_labels = df[["album_id", "mgg_database"] + existing_variables]
mutation_labels = mutation_labels.rename(columns={"mgg_database": "database_id"})
mutation_labels[existing_variables] = mutation_labels[existing_variables].replace({True: 1, False: 0})


# filter out samples that have been used in model training
df = pd.read_excel(BM_VARIABLE_TABLE_PATH)
mutation_labels = mutation_labels[
    ~mutation_labels.set_index(["album_id", "database_id"]).index.isin(
        df.set_index(["album_id", "database_id"]).index
    )
]
print(f"Found {len(mutation_labels)} BM samples not in model training")

# create dummy subject_id columns
mutation_labels["subject_id"] = mutation_labels.index

display(mutation_labels)

#if SAVE_FILES:
#    mutation_labels.to_excel(f"data/{DATE}/sweden_mutation_labels_{DATE}.xlsx")

### **Anonymized mutation labels**

In [ ]:
mutation_map = {
    "RTK_RAS_mutations": "A",
    "NPM1_OR_FLT3_ITD": "B",
    "CpG_mutations": "C",
    "FLT3_mutations": "D",
    "FLT3_ITD": "E",
    "NPM1": "F",
    "MDS_related_mutations": "G",
    "Transcription_mutations": "H",
    "DNMT3A": "I",
    "chromatin_cohesin_mutations": "J",
    "IDH_mutations": "K",
    "complex_karyotype": "L",
    "TET2": "M",
    "Spliceosome_mutations": "N",
    "monosomal_karyotype": "O",
    "RAS_mutations": "P",
    "IDH2": "Q",
    "trisomy_add_dup_amp_8": "R",
    "monosomy_del_7": "S",
    "TP53_OR_monosomy17_del_17p": "T",
    "monosomy5_del_5q": "U",
    "RUNX1": "V",
    "NRAS": "W",
    "SRSF2": "X",
    "TP53": "Y",
    "BCOR_mutations": "Z",
    "FLT3_TKD": "AA",
    "BCOR": "AB",
    "ASXL1": "AC",
    "RUNX1-RUNX1T1_OR_t_8_21": "AD",
    "monosomy17_del_17p": "AE",
    "WT1": "AF",
    "IDH1": "AG",
    "CEBPA": "AH",
    "PTPN11": "AI",
    "translocation11": "AJ",
    "KRAS": "AK",
    "JAK2": "AL",
    "GATA_mutations": "AM",
    "STAG2": "AN",
    "translocation3": "AO",
    "GATA2": "AP",
    "NF1": "AQ",
    "CBL": "AR",
    "PHF6": "AS",
    "U2AF1": "AT",
    "KIT": "AU",
    "KMT2A_rearrangement": "AV",
    "CBFB-MYH11": "AW",
    "EP300": "AX",
    "EZH2": "AY",
    "PML-RARA": "AZ"
}

In [ ]:
mutation_labels_anonymized =  mutation_labels.copy()
for variable in existing_variables:
    mutation_labels_anonymized = mutation_labels_anonymized.rename(columns={variable: mutation_map[variable]})

display(mutation_labels_anonymized.head())

if SAVE_FILES:
    V = DATA_VERSION_IDENTIFIER
    save_with_dummy(mutation_labels_anonymized, f"data/{V}/mutation_labels/sweden_bm_samples_{V}.xlsx", id_columns=["subject_id", "album_id", "database_id"])